# MobileNetV3-Small training

Дообучение MobileNetV3-Small (ImageNet-инициализация torchvision) на собственном датасете `combined_elongated_500k`:
350 000 train / 75 000 val / 75 000 test, классы сбалансированы. Метка `0` — обычная ориентация, `1` — поворот на 180°.
Источники: 350k реальных кропов из TextOCR и 150k синтетических строк (ru / en / смешанные). Подробнее — README, раздел «Данные».

Датасет скачать по ссылке из README и распаковать в `data/combined_elongated_500k/`. Окружение — `requirements-train.txt`;
для полного обучения нужен GPU. При `INITIALIZATION = 'imagenet'` torchvision скачает веса ImageNet при первом запуске;
без сети укажите локальный `IMAGENET_WEIGHTS` или выберите `INITIALIZATION = 'random'`.

Чекпоинт выбирается по Brier на OCR-части `val`; `test` используется один раз после обучения.
Каждый запуск пишет чекпоинт и отчёты в новую папку `runs/`. Итоговый чекпоинт этого запуска лежит в
`weights/mobilenet_v3_small_orientation.pt`, его логи — в `weights/mobilenet_run/`.


In [ ]:
import hashlib
import json
import random
import sys
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

repo_candidates = (Path.cwd(), *Path.cwd().parents)
ROOT = next((p for p in repo_candidates if (p / 'data/combined_elongated_500k/labels.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Не найден data/combined_elongated_500k/labels.csv; запустите ноутбук из репозитория')
sys.path.insert(0, str(ROOT))
from training.data import CropDataset, read_annotations
from training.metrics import grouped_metrics, sigmoid
from training.model import load_checkpoint, make_model, predict_logits, save_checkpoint

DATA_ROOT = ROOT / 'data/combined_elongated_500k'
CSV_PATH = DATA_ROOT / 'labels.csv'
SEED = 42
HEIGHT, WIDTH = 48, 320
BATCH_SIZE = 32
NUM_WORKERS = 4 if sys.platform != 'win32' else 0
EPOCHS, PATIENCE = 50, 10
ROTATE_PROB = 0.5
BACKBONE_LR, HEAD_LR = 1e-4, 1e-3
WEIGHT_DECAY = 1e-4
INITIALIZATION = 'imagenet'  # 'imagenet' или 'random'
IMAGENET_WEIGHTS = None  # Path('/path/to/torchvision_state_dict.pt') при работе без сети
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print(f'Данные: {DATA_ROOT} | устройство: {DEVICE} | torch: {torch.__version__}')

In [ ]:
rows, summary = read_annotations(CSV_PATH, DATA_ROOT)
parts = {split: [row for row in rows if row['split'] == split] for split in ('train', 'val', 'test')}
if sum(map(len, parts.values())) != len(rows):
    raise ValueError('Ожидаются только train, val и test')
if any(not parts[split] for split in parts):
    raise ValueError('Каждая часть train/val/test должна быть непустой')
if any(set(row['label'] for row in parts[split]) != {0, 1} for split in parts):
    raise ValueError('В каждой части должны быть оба класса')
if not any(row['source'] == 'ocr' for row in parts['val']):
    raise ValueError('В val нет OCR-кропов для выбора checkpoint')
manifest = json.loads((DATA_ROOT / 'manifest.json').read_text(encoding='utf-8'))
if len(rows) != manifest['total_count']:
    raise ValueError('Количество строк не совпадает с manifest.json')
for split, split_rows in parts.items():
    labels = Counter(row['label'] for row in split_rows)
    sources = Counter(row['source'] for row in split_rows)
    print(f'{split}: {len(split_rows):,}; классы {dict(labels)}; источники {dict(sources)}')
print('Проверены пути, метки и отсутствие пересечения групп между частями')

In [ ]:
# Кропы приводятся к размеру 48×320 с сохранением пропорций и симметричными полями. 
# Дополнительный поворот train-изображения на 180° одновременно меняет метку; 
# для val/test аугментации нет.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
datasets = {split: CropDataset(split_rows, HEIGHT, WIDTH, training=(split == 'train'),
                               rotate_prob=ROTATE_PROB, seed=SEED)
            for split, split_rows in parts.items()}
loaders = {split: DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=(split == 'train'),
                             num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
                             persistent_workers=False,
                             generator=torch.Generator().manual_seed(SEED + index))
           for index, (split, dataset) in enumerate(datasets.items())}
sample_images, sample_labels = next(iter(loaders['train']))
print('Размер батча:', tuple(sample_images.shape), '| метки:', sample_labels[:8].tolist())

In [ ]:
if INITIALIZATION not in ('imagenet', 'random'):
    raise ValueError('INITIALIZATION: imagenet или random')
if IMAGENET_WEIGHTS is not None and INITIALIZATION != 'imagenet':
    raise ValueError('Локальные ImageNet-веса требуют INITIALIZATION = imagenet')
if IMAGENET_WEIGHTS is not None and not Path(IMAGENET_WEIGHTS).is_file():
    raise FileNotFoundError(IMAGENET_WEIGHTS)
model = make_model(INITIALIZATION, IMAGENET_WEIGHTS).to(DEVICE)
optimizer = torch.optim.AdamW([
    {'params': model.features.parameters(), 'lr': BACKBONE_LR},
    {'params': model.classifier.parameters(), 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
print('Параметров модели:', sum(parameter.numel() for parameter in model.parameters()))

In [ ]:
# Обучение сохраняет лучший checkpoint по Brier score на реальных OCR-кропах `val`. 
# После каждой эпохи в `history.json` остаются метрики для всей `val` и отдельно для OCR/синтетики. 
RUN_DIR = ROOT / 'runs' / f"mobilenet_combined_{datetime.now():%Y%m%d_%H%M%S_%f}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
csv_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()
config = dict(seed=SEED, height=HEIGHT, width=WIDTH, batch_size=BATCH_SIZE,
              workers=NUM_WORKERS, epochs=EPOCHS, patience=PATIENCE,
              rotate_prob=ROTATE_PROB, backbone_lr=BACKBONE_LR, head_lr=HEAD_LR,
              weight_decay=WEIGHT_DECAY, initialization=INITIALIZATION,
              imagenet_weights=str(IMAGENET_WEIGHTS) if IMAGENET_WEIGHTS else None,
              device=str(DEVICE), torch_version=str(torch.__version__),
              labels_sha256=csv_sha256, selection='source=ocr:val_brier')
(RUN_DIR / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
(RUN_DIR / 'data_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
history = []
best_brier, best_epoch, stale = float('inf'), 0, 0
for epoch in range(1, EPOCHS + 1):
    datasets['train'].epoch = epoch
    model.train()
    total_loss, seen = 0.0, 0
    start = time.perf_counter()
    for step, (images, targets) in enumerate(loaders['train'], 1):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(images).squeeze(1)
            loss = criterion(logits, targets)
        if not torch.isfinite(loss):
            raise RuntimeError('Неконечная функция потерь; проверьте данные или уменьшите learning rate')
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(targets)
        seen += len(targets)
        if step % 100 == 0:
            print(f'Эпоха {epoch}: {seen:,}/{len(datasets["train"]):,}, loss={total_loss/seen:.4f}', flush=True)
    val_logits = predict_logits(model, loaders['val'], DEVICE)
    val_metrics = grouped_metrics(parts['val'], sigmoid(val_logits))
    ocr_brier = val_metrics['source=ocr']['brier']
    record = dict(epoch=epoch, train_loss=total_loss/seen, val=val_metrics,
                  train_seconds=time.perf_counter()-start)
    history.append(record)
    (RUN_DIR / 'history.json').write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Эпоха {epoch}: OCR val Brier={ocr_brier:.5f}; all val Brier={val_metrics["all"]["brier"]:.5f}')
    if ocr_brier < best_brier:
        best_brier, best_epoch, stale = ocr_brier, epoch, 0
        save_checkpoint(RUN_DIR / 'best_raw.pt', model, HEIGHT, WIDTH,
                        epoch=epoch, validation_brier=ocr_brier,
                        training_annotation_sha256=csv_sha256, selected_scope='source=ocr')
    else:
        stale += 1
    scheduler.step()
    if stale >= PATIENCE:
        print('Ранняя остановка')
        break
print(f'Лучшая эпоха: {best_epoch}; OCR val Brier: {best_brier:.5f}')
print('Checkpoint:', RUN_DIR / 'best_raw.pt')

## Экспорт в ONNX

Инференс (`solution.ipynb`) использует ONNX-версию модели, чтобы не ставить torch рядом с paddlepaddle-gpu:

```bash
python -m training.export_onnx --checkpoint weights/mobilenet_v3_small_orientation.pt \
    --out weights/mobilenet_v3_small_orientation.onnx
```